# Gemini 비동기 요청

일반 API에서 배운 코루틴과 `asyncio.gather()`를 Gemini API 요청에 적용합니다.

공식 SDK 문서: https://googleapis.github.io/python-genai/

### LLM API에서 비동기를 사용하는 이유

LLM API는 요청을 보낸 뒤 모델이 답변을 생성할 때까지 기다리는 시간이 깁니다. 서로 독립적인 질문을 하나씩 요청하면 첫 번째 답변이 끝난 뒤에야 두 번째 요청을 시작하므로 전체 대기 시간이 길어집니다.

비동기로 여러 요청을 시작하면 한 요청의 응답을 기다리는 동안 다른 요청도 처리될 수 있습니다. 예를 들어 문서 여러 개를 각각 요약하거나, 여러 상품의 설명을 생성하거나, 여러 사용자의 질문을 처리할 때 전체 처리 시간을 줄일 수 있습니다.

다만 비동기는 **한 요청의 답변 생성 속도를 높이는 기능이 아닙니다.** 여러 독립적인 요청의 대기 시간을 겹쳐서 전체 처리 시간을 줄이는 방법입니다.

FastAPI 같은 웹 서버에서도 비동기 LLM 호출이 중요합니다. `async def` 엔드포인트에서 LLM 응답을 `await`하면, 응답을 기다리는 동안 이벤트 루프가 다른 사용자의 요청을 처리할 수 있어 서버의 동시 처리 효율이 높아집니다. 반대로 비동기 엔드포인트 안에서 동기 LLM 클라이언트를 직접 호출하면 이벤트 루프를 막을 수 있으므로 비동기 클라이언트를 사용하는 것이 적절합니다.

동시에 너무 많은 요청을 보내면 API 호출 한도에 걸릴 수 있고 비용도 요청 수만큼 발생합니다. 따라서 실제 서비스에서는 세마포어 또는 작업 큐를 사용해 동시 요청 개수를 제한해야 합니다.

### 실습 환경 준비

In [2]:
import asyncio
import os
import time

from dotenv import load_dotenv
from google import genai

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError(".env 파일에 GEMINI_API_KEY를 설정해 주세요.")

model = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")
client = genai.Client(api_key=api_key)
async_client = client.aio

### 동기 요청과 비동기 요청 비교

동기 요청은 `client.interactions.create()`를 바로 호출합니다. 비동기 요청은 `client.aio.interactions.create()`를 `await`합니다. 두 방식은 응답을 기다리는 방법만 다르고 요청에 전달하는 주요 인자는 같습니다.

In [3]:
interaction = await async_client.interactions.create(
    model=model,
    input="비동기 요청을 한 문장으로 설명해 줘.",
    store=False,
)

print(interaction.output_text)

비동기 요청이란 **요청을 보내고 응답이 올 때까지 기다리지 않고, 그동안 다른 작업을 계속 처리하는 방식**입니다.


### Gemini 요청을 코루틴 함수로 만들기

프롬프트를 받아 모델의 답변을 반환하는 코루틴 함수를 만듭니다. 호출하면 코루틴 객체가 만들어지고, `await`할 때 결과를 받을 수 있습니다.

In [4]:
async def ask_gemini(prompt: str) -> str:
    interaction = await async_client.interactions.create(
        model=model,
        input=prompt,
        store=False,
    )
    return interaction.output_text

### 여러 LLM 요청을 순서대로 실행하기

각 요청을 하나씩 `await`하면 앞 요청이 끝난 후 다음 요청을 시작합니다.

In [5]:
prompts = [
    "LLM을 한 문장으로 설명해 줘.",
    "임베딩을 한 문장으로 설명해 줘.",
    "토큰을 한 문장으로 설명해 줘.",
]

start = time.perf_counter()
sequential_answers = []
for prompt in prompts:
    answer = await ask_gemini(prompt)
    sequential_answers.append(answer)

print(f"순차 실행 시간: {time.perf_counter() - start:.2f}초")
for answer in sequential_answers:
    print(answer)

순차 실행 시간: 17.97초
**거대 언어 모델(LLM)은 방대한 양의 텍스트 데이터를 학습하여 인간처럼 자연스럽게 언어를 이해하고, 질문에 답하거나 글을 작성하는 인공지능 기술입니다.**
**임베딩은 사람이 사용하는 언어나 데이터의 '의미'를 컴퓨터가 이해하고 계산할 수 있도록 '숫자의 배열(벡터)'로 변환하는 기술입니다.**
**토큰은 AI가 텍스트를 처리하고 이해하기 위해 나누는 가장 작은 데이터 단위입니다.** 

*(참고: 블록체인 분야에서는 특정 자산이나 권리를 나타내는 '디지털 증표'를 의미하기도 합니다.)*


### 여러 LLM 요청을 함께 실행하기

`asyncio.gather()`를 사용하면 한 요청의 응답을 기다리는 동안 다른 요청도 진행할 수 있습니다. 결과는 전달한 프롬프트 순서대로 반환됩니다.

동시 요청이 지나치게 많으면 API 사용량 제한에 걸릴 수 있으므로 실제 서비스에서는 요청 개수를 제한해야 합니다. 이 수업에서는 세 개의 요청만 사용합니다.

In [6]:
start = time.perf_counter()
concurrent_answers = await asyncio.gather(
    ask_gemini(prompts[0]),
    ask_gemini(prompts[1]),
    ask_gemini(prompts[2]),
)

print(f"함께 실행한 시간: {time.perf_counter() - start:.2f}초")
for prompt, answer in zip(prompts, concurrent_answers):
    print("질문:", prompt)
    print("답변:", answer)

함께 실행한 시간: 6.77초
질문: LLM을 한 문장으로 설명해 줘.
답변: **LLM(거대 언어 모델)은 방대한 양의 텍스트 데이터를 학습하여 인간처럼 맥락을 이해하고 자연스럽게 대화하며 글을 작성하는 인공지능입니다.**
질문: 임베딩을 한 문장으로 설명해 줘.
답변: **임베딩은 컴퓨터가 데이터의 의미와 맥락을 이해할 수 있도록, 텍스트나 이미지 같은 정보의 특징을 다차원 공간의 숫자(벡터)로 변환하는 기술입니다.**
질문: 토큰을 한 문장으로 설명해 줘.
답변: **토큰은 인공지능이나 컴퓨터가 텍스트와 데이터를 효율적으로 이해하고 처리하기 위해 나누어 놓은 '가장 작은 정보 단위'입니다.**


### 실습 문제

세 가지 Python 주제를 Gemini에 동시에 질문하세요.

- `async def explain_python(topic)` 코루틴 함수를 작성합니다.
- `list comprehension`, `generator`, `decorator`를 각각 두 문장으로 설명하도록 요청합니다.
- 세 요청을 `asyncio.gather()`로 실행합니다.
- 주제와 답변을 함께 출력합니다.
- 전체 실행 시간을 출력합니다.

In [10]:
# TODO: explain_python 코루틴 함수를 작성하세요.
prompts = [
    "list comprehension 두 문장으로 설명해줘.",
    "generator 두 문장으로 설명해줘.",
    "decorator 두 문장으로 설명해줘.",
]
start = time.perf_counter()
concurrent_answers = await asyncio.gather(
    ask_gemini(prompts[0]),
    ask_gemini(prompts[1]),
    ask_gemini(prompts[2]),
)

print(f"함께 실행한 시간: {time.perf_counter() - start:.2f}초")
print()
for prompt, answer in zip(prompts, concurrent_answers):
    print("주제", prompt)
    print("답변:", answer)
    print()

# TODO: 세 주제를 함께 요청하고 결과와 실행 시간을 출력하세요.


함께 실행한 시간: 5.60초

주제 list comprehension 두 문장으로 설명해줘.
답변: **리스트 컴프리핸션(List Comprehension)**은 파이썬에서 대괄호`[]` 내부에 `for`문과 `if`문을 조합해 새로운 리스트를 한 줄로 간결하게 만드는 생성 문법입니다. 

이를 통해 기존의 여러 줄짜리 반복문 코드를 획기적으로 줄여 코드의 가독성과 작성 효율을 높일 수 있습니다.

주제 generator 두 문장으로 설명해줘.
답변: 제너레이터는 모든 데이터를 한 번에 메모리에 올리지 않고, 필요할 때마다 순차적으로 값을 하나씩 만들어내는 특수한 함수입니다. 

이를 통해 대용량 데이터를 처리할 때 메모리 사용량을 크게 줄이고 프로그램의 실행 효율을 높일 수 있습니다.

주제 decorator 두 문장으로 설명해줘.
답변: 데코레이터는 기존 함수의 코드를 직접 수정하지 않고도 새로운 기능을 추가하거나 동작을 변경할 수 있게 해주는 기능입니다. 

주로 대상 함수를 인자로 받아 앞뒤에 로깅, 권한 검사 등의 부가적인 작업을 수행하도록 감싸는(wrap) 역할로 사용됩니다.



In [11]:
prompts = [
    "list comprehension",
    "generator",
    "decorator",
]


async def explain_python(topic):
    answer = await ask_gemini(f"{topic} 두 문장으로 설명해줘.")
    return answer


start = time.perf_counter()

answers = await asyncio.gather(
    explain_python(prompts[0]),
    explain_python(prompts[1]),
    explain_python(prompts[2]),
)

print(f"함께 실행한 시간: {time.perf_counter() - start:.2f}초")

for topic, answer in zip(prompts, answers):
    print("주제:", topic)
    print("답변:", answer)
    print()

함께 실행한 시간: 5.70초
주제: list comprehension
답변: **리스트 컴프리헨션은 대괄호`[]` 안에 반복문과 조건문을 한 줄로 작성하여 새로운 리스트를 간결하게 생성하는 파이썬 문법입니다.**

**기존의 `for` 루프를 사용하는 것보다 코드 길이를 줄여주어 가독성을 높이고 작성 효율성을 크게 향상시킵니다.**

주제: generator
답변: 제너레이터는 모든 데이터를 한 번에 메모리에 올리지 않고, 필요할 때마다 값을 하나씩 순차적으로 생성하는 특수한 함수입니다. 

이 방식을 통해 대용량 데이터를 처리할 때 메모리를 획기적으로 절약하고 프로그램의 실행 효율을 높일 수 있습니다.

주제: decorator
답변: 데코레이터는 기존 함수의 코드를 직접 수정하지 않고도 새로운 기능을 추가하거나 확장할 수 있게 해주는 파이썬의 문법입니다. 

`@` 기호를 사용해 여러 함수에 로깅, 권한 검사, 실행 시간 측정 같은 공통 작업을 손쉽게 재사용할 수 있습니다.

